In [1]:
# @title 1. Installation of Required Libraries
# This first cell handles the installation of all necessary Python libraries.
# Added onnxruntime-gpu to support inference with the exported ONNX model.
!pip install -q ultralytics pandas numpy opencv-python-headless matplotlib seaborn albumentations kaggle kagglehub onnxruntime-gpu


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# @title 2. (Optional) Download Dataset from Kaggle
# This cell provides an option to download the dataset directly from Kaggle.
# If you already have the 'Dataset' folder, you can set DOWNLOAD_FROM_KAGGLE to False.

import os
import json
import zipfile
import shutil

DOWNLOAD_FROM_KAGGLE = False # @param {type:"boolean"}

if DOWNLOAD_FROM_KAGGLE:
    print("Attempting to download dataset from Kaggle...")
    
    # --- Setup Kaggle API Token ---
    api_token = {"username":"tomeratar","key":"481b7c26190e8c6481146d7acf863ff9"}
    
    kaggle_dir = os.path.expanduser("~/.kaggle")
    os.makedirs(kaggle_dir, exist_ok=True)
    with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as file:
        json.dump(api_token, file)
    os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600) # Set correct permissions
    
    # --- Download and Unzip ---
    import kagglehub
    
    output_dir = "Dataset"
    if os.path.exists(output_dir):
        print(f"Removing existing '{output_dir}' directory...")
        shutil.rmtree(output_dir)

    print("Downloading dataset from goelyash/animal-dataset...")
    download_path = kagglehub.dataset_download("goelyash/animal-dataset")
    print(f"Dataset downloaded to: {download_path}")

    # --- Verification and Cleanup ---
    try:
        with zipfile.ZipFile(download_path, 'r') as zip_ref:
            # Get the list of files in the zip to verify against later
            file_list = zip_ref.namelist()
            total_files_in_zip = len(file_list)
            print(f"Archive contains {total_files_in_zip} files.")
            
            print(f"Unzipping dataset to '{output_dir}' folder...")
            zip_ref.extractall(output_dir)

        # Count the number of files actually extracted
        extracted_files_count = 0
        for root, dirs, files in os.walk(output_dir):
            extracted_files_count += len(files)
        
        print(f"Successfully extracted {extracted_files_count} files.")

        if extracted_files_count == total_files_in_zip and extracted_files_count > 0:
             print("Verification successful: Extracted file count matches archive file count.")
             print(f"Deleting downloaded zip file: {download_path}")
             os.remove(download_path)
        else:
            error_message = (f"ERROR: Dataset extraction failed. "
                           f"Expected {total_files_in_zip} files, but found {extracted_files_count}. "
                           "This might be due to a lack of disk space. Terminating the notebook run.")
            print(error_message)
            raise RuntimeError(error_message)

    except zipfile.BadZipFile:
        error_message = "ERROR: The downloaded file is not a valid zip archive or is corrupted. Please try the download again."
        print(error_message)
        raise RuntimeError(error_message)
    
    BASE_PATH = output_dir + "/"

else:
    print("Skipping Kaggle download.")
    BASE_PATH = "Dataset/"

if not os.path.exists(BASE_PATH):
     raise FileNotFoundError(f"The specified BASE_PATH '{BASE_PATH}' does not exist. Please check the path or enable the Kaggle download.")
else:
    print(f"Dataset base path set to: {BASE_PATH}")

Skipping Kaggle download.
Dataset base path set to: Dataset/


In [3]:
# @title 3. Importing Libraries and Initial Setup
import pandas as pd
import numpy as np
import cv2
from tqdm.notebook import tqdm
import yaml
from ultralytics import YOLO
import matplotlib.pyplot as plt

# --- Configuration ---
TRAIN_ANNOTATION_PATH = os.path.join(BASE_PATH, "Annotation/Train_annotation.csv")
VAL_ANNOTATION_PATH = os.path.join(BASE_PATH, "Annotation/Val_annotation.csv")
CLASS_NAME_PATH = os.path.join(BASE_PATH, "Class_name.csv")

# --- Define output directories ---
TRAIN_LABEL_DIR = "yolo_data/labels/train"
VAL_LABEL_DIR = "yolo_data/labels/val"
TRAIN_IMAGE_DIR = "yolo_data/images/train"
VAL_IMAGE_DIR = "yolo_data/images/val"

# --- Clean up and create directories ---
if os.path.exists("yolo_data"):
    shutil.rmtree("yolo_data")

os.makedirs(TRAIN_LABEL_DIR, exist_ok=True)
os.makedirs(VAL_LABEL_DIR, exist_ok=True)
os.makedirs(TRAIN_IMAGE_DIR, exist_ok=True)
os.makedirs(VAL_IMAGE_DIR, exist_ok=True)

print("Directories created successfully.")

Directories created successfully.


In [4]:
# @title 4. Define the List of Farm/Rural Area Animals
# This list is narrowed down to 10 animals most likely to be damaging
# to farms or people in rural areas, excluding birds, insects, and fish.

ANIMALS_TO_TRAIN_LOWERCASE = [
    'cougar',
    'coyote',
    'brown bear',
    'wild boar',
    'warthog',
    'indian cobra',
    'lion',
    'African elephant',
    'hyena'
]

# This part remains the same, ensuring correct capitalization is used.
class_df = pd.read_csv(CLASS_NAME_PATH)
correct_case_map = {name.lower(): name for name in class_df['Class_name']}
SELECTED_ANIMALS = [correct_case_map.get(animal, animal) for animal in ANIMALS_TO_TRAIN_LOWERCASE]

# The class-to-integer mapping is updated for the new list.
selected_class_to_int = {name: i for i, name in enumerate(SELECTED_ANIMALS)}

# The print statement is updated for clarity.
print(f"Total number of classes to train on: {len(SELECTED_ANIMALS)}")
print("Classes:", SELECTED_ANIMALS)

Total number of classes to train on: 9
Classes: ['cougar', 'coyote', 'brown bear', 'wild boar', 'warthog', 'Indian cobra', 'lion', 'African elephant', 'hyena']


In [5]:
# @title 5. Data Preprocessing Function (Fixed for KeyError)
import pandas as pd
from tqdm.notebook import tqdm
import os
import shutil

def process_annotations(csv_path, image_dir, label_dir, is_train=True):
    # This prevents the DtypeWarning you may have seen earlier
    df = pd.read_csv(csv_path, low_memory=False)

    # --- FIX: Dynamically find the max number of objects ---
    # This prevents the KeyError by checking how many 'Name_X' columns actually exist in the CSV.
    name_cols_present = [col for col in df.columns if col.startswith('Name_')]
    if not name_cols_present:
        print(f"Warning: No 'Name_X' columns found in {os.path.basename(csv_path)}. Skipping this file.")
        return
    max_objects = max(int(col.split('_')[1]) for col in name_cols_present)
    # --- End of Fix ---

    # The rest of the code now uses the dynamically found max_objects value
    name_cols = [f'Name_{i}' for i in range(1, max_objects + 1)]
    xmin_cols = [f'Xmin_{i}' for i in range(1, max_objects + 1)]
    ymin_cols = [f'Ymin_{i}' for i in range(1, max_objects + 1)]
    xmax_cols = [f'Xmax_{i}' for i in range(1, max_objects + 1)]
    ymax_cols = [f'Ymax_{i}' for i in range(1, max_objects + 1)]

    files_copied = 0
    for index, row in tqdm(df.iterrows(), total=df.shape[0], desc=f"Processing {'Train' if is_train else 'Val'} data"):
        if is_train:
            parts = row['File_location'].split('/')
            if len(parts) != 2:
                continue
            class_id, filename_stem = parts
            src_img_path = os.path.join(BASE_PATH, 'Train', class_id, f"{filename_stem}.JPEG")
        else:
            src_img_path = os.path.join(BASE_PATH, 'val', 'val', f"{row['File_location'].split('/')[-1]}.JPEG")

        if not os.path.exists(src_img_path):
            continue

        img_width = row['Width']
        img_height = row['Height']
        yolo_annotations = []

        for i in range(max_objects):
            class_name = row[name_cols[i]]
            if pd.isna(class_name) or class_name == '0' or class_name not in SELECTED_ANIMALS:
                continue

            class_int = selected_class_to_int[class_name]
            xmin, ymin, xmax, ymax = row[xmin_cols[i]], row[ymin_cols[i]], row[xmax_cols[i]], row[ymax_cols[i]]

            box_width = (xmax - xmin) / img_width
            box_height = (ymax - ymin) / img_height
            center_x = (xmin + (xmax - xmin) / 2) / img_width
            center_y = (ymin + (ymax - ymin) / 2) / img_height
            yolo_annotations.append(f"{class_int} {center_x} {center_y} {box_width} {box_height}")

        if yolo_annotations:
            img_filename = os.path.basename(src_img_path)
            label_filename = os.path.splitext(img_filename)[0] + ".txt"
            with open(os.path.join(label_dir, label_filename), 'w') as f:
                f.write("\n".join(yolo_annotations))
            shutil.copy(src_img_path, os.path.join(image_dir, img_filename))
            files_copied += 1

    print(f"Process complete. A total of {files_copied} images were successfully processed and copied.")

print("Annotation processing function defined.")

Annotation processing function defined.


In [6]:
# @title 5.5. (NEW) Create a Robust 80/20 Data Split
# This cell combines, filters, shuffles, and re-splits the dataset
# to create a more reliable validation set, which is crucial for
# trustworthy early stopping.

import pandas as pd

# --- 1. Load and combine the original annotation files ---
print("Loading original annotation files...")
train_df = pd.read_csv(TRAIN_ANNOTATION_PATH, low_memory=False)
val_df = pd.read_csv(VAL_ANNOTATION_PATH, low_memory=False)
combined_df = pd.concat([train_df, val_df], ignore_index=True)
print(f"Combined annotations: {len(combined_df)} rows.")

# --- 2. Filter for images containing only your selected animals ---
# This ensures the split is based only on the data you're actually using.
print("Filtering for rows containing selected animals...")
name_cols = [col for col in combined_df.columns if col.startswith('Name_')]
filtered_rows = combined_df[name_cols].isin(SELECTED_ANIMALS).any(axis=1)
relevant_df = combined_df[filtered_rows].copy() # Use .copy() to avoid SettingWithCopyWarning
print(f"Found {len(relevant_df)} relevant rows.")

# --- 3. Shuffle and split the data ---
print("Shuffling and splitting data into 80/20 ratio...")
shuffled_df = relevant_df.sample(frac=1, random_state=42) # random_state for reproducibility
split_index = int(0.8 * len(shuffled_df))

new_train_df = shuffled_df.iloc[:split_index]
new_val_df = shuffled_df.iloc[split_index:]

# --- 4. Save the new splits to temporary files ---
# These new CSVs will be used for processing in the next step.
NEW_TRAIN_ANNOTATION_PATH = "new_train_annotations.csv"
NEW_VAL_ANNOTATION_PATH = "new_val_annotations.csv"
new_train_df.to_csv(NEW_TRAIN_ANNOTATION_PATH, index=False)
new_val_df.to_csv(NEW_VAL_ANNOTATION_PATH, index=False)

print(f"\nData successfully re-split.")
print(f"New training set size: {len(new_train_df)}")
print(f"New validation set size: {len(new_val_df)}")

Loading original annotation files...
Combined annotations: 237960 rows.
Filtering for rows containing selected animals...
Found 5569 relevant rows.
Shuffling and splitting data into 80/20 ratio...

Data successfully re-split.
New training set size: 4455
New validation set size: 1114


In [7]:
# @title 6. Process Training and Validation Datasets
# (MODIFIED) This cell now uses the newly created 80/20 annotation splits.

print("Starting to process the new training data split...")
process_annotations(NEW_TRAIN_ANNOTATION_PATH, TRAIN_IMAGE_DIR, TRAIN_LABEL_DIR, is_train=True)
print("\nNew training data processed successfully.")

print("\nStarting to process the new validation data split...")
process_annotations(NEW_VAL_ANNOTATION_PATH, VAL_IMAGE_DIR, VAL_LABEL_DIR, is_train=False)
print("\nNew validation data processed successfully.")

Starting to process the new training data split...


Processing Train data:   0%|          | 0/4455 [00:00<?, ?it/s]

Process complete. A total of 4098 images were successfully processed and copied.

New training data processed successfully.

Starting to process the new validation data split...


Processing Val data:   0%|          | 0/1114 [00:00<?, ?it/s]

Process complete. A total of 93 images were successfully processed and copied.

New validation data processed successfully.


In [8]:
# @title 7. Create the `dataset.yaml` File

# The dictionary now uses the new list of 10 selected animals.
dataset_yaml_content = {
    'train': os.path.abspath(TRAIN_IMAGE_DIR),
    'val': os.path.abspath(VAL_IMAGE_DIR),
    'nc': len(SELECTED_ANIMALS),
    'names': SELECTED_ANIMALS
}

YAML_PATH = 'dataset.yaml'
with open(YAML_PATH, 'w') as f:
    yaml.dump(dataset_yaml_content, f, default_flow_style=False)

print(f"'{YAML_PATH}' created successfully!")
print("\n--- YAML File Content ---")
with open(YAML_PATH, 'r') as f:
    print(f.read())
print("------------------------")

'dataset.yaml' created successfully!

--- YAML File Content ---
names:
- cougar
- coyote
- brown bear
- wild boar
- warthog
- Indian cobra
- lion
- African elephant
- hyena
nc: 9
train: C:\Training\yolo_data\images\train
val: C:\Training\yolo_data\images\val

------------------------


# 8. Augmentation Strategy
With ~1300 images per class, a strong augmentation strategy is crucial for robustness. The `ultralytics` library has powerful, built-in augmentations that are applied on-the-fly during training. We will use the default YOLO augmentation settings, which are highly optimized for performance.

In [9]:
# @title 9. Optimized Model Training (YOLOv10m Version with ONNX Export)
# (MODIFIED) This cell includes stronger augmentation and regularization parameters.

# --- 1. Select the Base Model ---
model_version = 'yolov10m.pt'
model = YOLO(model_version)

# --- 2. Define a Unique & Accurate Name for the Training Run (IMPROVEMENT) ---
# The name now correctly reflects the model (m) and epoch count (400).
run_name = f'yolo_dangerous_animal_detection_m_400_epochs'

print(f"\n{'='*20} TRAINING {model_version.upper()} {'='*20}")
print(f"This training uses the medium 'm' model for up to 400 epochs.")

# --- 3. Start Training with Optimized Parameters ---
model.train(
    data=YAML_PATH,
    epochs=400,
    imgsz=640,
    batch=-1,
    patience=25,
    name=run_name,
    exist_ok=True,
    format='onnx',

    # --- NEW: Stronger Augmentation & Regularization ---
    augment=True,      # Keep this enabled
    degrees=25.0,      # Add random rotation
    translate=0.15,    # Add random translation
    scale=0.6,         # Add random scaling/zoom
    mixup=0.1,         # Add mixup augmentation (blends images)
    dropout=0.2        # Add dropout for regularization
)

print(f"\n--- Finished Training {model_version.upper()} ---")


==================== TRAINING YOLOV10M.PT ====================
This training uses the medium 'm' model for up to 400 epochs.
New https://pypi.org/project/ultralytics/8.3.203 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.184  Python-3.13.3 torch-2.8.0+cu126 CUDA:0 (GRID P40-8Q, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=25.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.2, dynamic=False, embed=None, epochs=400, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=onnx, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov10m.pt, 

train: Scanning C:\Training\yolo_data\labels\train... 4098 images, 0 backgrounds, 0 corrupt: 100%|██████████| 4098/4098


train: New cache created: C:\Training\yolo_data\labels\train.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch: CUDA:0 (GRID P40-8Q) 8.00G total, 0.19G reserved, 0.18G allocated, 7.63G free
      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output
    16494550       64.02         1.594         81.36           nan        (1, 3, 640, 640)                    list
    16494550         128         3.290         87.59           nan        (2, 3, 640, 640)                    list
    16494550       256.1         5.782         133.2           nan        (4, 3, 640, 640)                    list
    16494550       512.2        10.945          2128           nan        (8, 3, 6

train: Scanning C:\Training\yolo_data\labels\train.cache... 4098 images, 0 backgrounds, 0 corrupt: 100%|██████████| 409

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


...

WARNING Model does not support 'augment=True', reverting to single-scale prediction.
WARNING Model does not support 'augment=True', reverting to single-scale prediction.


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95):  88%|████████▊ | 14/16 [00:01

WARNING Model does not support 'augment=True', reverting to single-scale prediction.
WARNING Model does not support 'augment=True', reverting to single-scale prediction.


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 16/16 [00:01


                   all         93        121      0.882      0.883      0.917      0.821
                cougar          9         10      0.957        0.7      0.844      0.761
                coyote         11         12      0.997          1      0.995      0.872
            brown bear         11         12      0.769      0.917      0.975      0.892
             wild boar         11         24      0.879      0.708      0.857      0.703
               warthog          9         15      0.742      0.957      0.873      0.764
          Indian cobra          8          9          1      0.861      0.911      0.814
                  lion         11         15      0.753      0.867       0.85      0.754
      African elephant          8          9      0.931          1      0.995      0.926
                 hyena         15         15      0.906      0.933      0.956        0.9
Speed: 0.4ms preprocess, 11.2ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to runs\detec

In [10]:
# @title 10. Validation of the Trained Model (Modified)

import pandas as pd
import os
from ultralytics import YOLO

# (BUG FIX) Initialize a list to store metrics. This fixes the NameError in the next cell.
all_metrics = []

# --- 1. Define the path to the trained model's weights ---
# This must match the 'run_name' from the training cell above.
run_name = 'yolo_dangerous_animal_detection_m_400_epochs'
best_model_path = f'runs/detect/{run_name}/weights/best.pt'

# --- 2. Check if the model exists and run validation ---
if os.path.exists(best_model_path):
    print(f"--- Validating model: {best_model_path} ---")
    model = YOLO(best_model_path)
    metrics = model.val()

    # --- 3. Display the validation metrics in a clean format ---
    print("\n--- Validation Results ---")
    metrics_data = {
        # (CORRECTNESS) Dynamically get model name instead of hardcoding.
        'Model': model_version.replace('.pt','').upper(),
        'mAP50-95': metrics.box.map,
        'mAP50': metrics.box.map50,
        'mAP75': metrics.box.map75,
        'Precision': metrics.box.p[0],
        'Recall': metrics.box.r[0]
    }
    
    # (BUG FIX) Append the results to the list for the summary cell.
    all_metrics.append(metrics_data)
    
    results_df = pd.DataFrame([metrics_data])
    print(results_df.to_string(index=False))

else:
    print(f"\nERROR: Could not find trained weights at '{best_model_path}'.")
    print("Please ensure the training in Cell 9 completed successfully.")

--- Validating model: runs/detect/yolo_dangerous_animal_detection_m_400_epochs/weights/best.pt ---
Ultralytics 8.3.184  Python-3.13.3 torch-2.8.0+cu126 CUDA:0 (GRID P40-8Q, 8192MiB)
YOLOv10m summary (fused): 136 layers, 15,318,379 parameters, 0 gradients, 58.9 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 907.2361.6 MB/s, size: 259.9 KB)


val: Scanning C:\Training\yolo_data\labels\val.cache... 93 images, 0 backgrounds, 0 corrupt: 100%|██████████| 93/93 [00
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:10<0


                   all         93        121      0.889      0.877      0.921      0.824
                cougar          9         10      0.972        0.7       0.85      0.767
                coyote         11         12          1      0.989      0.995      0.874
            brown bear         11         12      0.773      0.917      0.975      0.891
             wild boar         11         24      0.886      0.708      0.869      0.711
               warthog          9         15      0.745          1      0.887      0.772
          Indian cobra          8          9          1      0.776      0.911      0.816
                  lion         11         15      0.743      0.867      0.851      0.752
      African elephant          8          9      0.935          1      0.995      0.928
                 hyena         15         15      0.947      0.933      0.957      0.906
Speed: 1.7ms preprocess, 12.3ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to runs\detec

In [11]:
# @title 11. Results Summary
# This cell displays a summary table comparing the performance of all trained models.

if all_metrics:
    results_df = pd.DataFrame(all_metrics)
    results_df = results_df.set_index('Model')
    print("\n--- Performance Comparison of YOLOv10 Models ---")
    print(results_df)
else:
    print("\nNo validation metrics were collected. Please ensure the training completed successfully.")


--- Performance Comparison of YOLOv10 Models ---
          mAP50-95     mAP50     mAP75  Precision  Recall
Model                                                    
YOLOV10M  0.824133  0.921277  0.899136   0.971835     0.7
